In [7]:
import os, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np, kagglehub
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from aijack.defense import GeneralMomentAccountant, DPSGDManager

# --- PARAMETRI DI TESI ---
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
batch_size = 32
lot_size = 32  # Nelle GAN stabili, lot_size = batch_size
lat_dim = 100
clients = 3
epochs = 10

# Parametri Privacy (da doc 8.2 e 8.4)
sigma = 0.5
l2_norm_clip = 1.0
t_delta = 1e-5

# --- DATASET PREPARATION ---
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=1), # Forza 1 canale come in doc 8.2.1
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
indices = np.random.permutation(len(full_dataset))
split = len(full_dataset) // clients
client_loaders = [DataLoader(Subset(full_dataset, indices[i*split : (i+1)*split]), 
                  batch_size=batch_size, shuffle=True) for i in range(clients)]

In [8]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(lat_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False), 
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, img): return self.main(img).view(-1, 1)

global_generator = Generator().to(device)
local_discriminators = [Discriminator().to(device) for _ in range(clients)]
gen_opt = optim.Adam(global_generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [9]:
accountant = GeneralMomentAccountant(noise_type="Gaussian", backend="python")

privacy_manager = DPSGDManager(
    accountant,
    optim.Adam,
    l2_norm_clip=l2_norm_clip,
    dataset=full_dataset,
    lot_size=lot_size,
    batch_size=batch_size,
    iterations=epochs * (len(full_dataset) // lot_size),
    # INTEGRAZIONE DPLIS (doc 8.4)
    smoothing=True,
    smoothing_radius=0.1
)

# Privatizzazione (ritorna la classe dell'ottimizzatore e i loader speciali)
dpoptimizer_cls, lot_loader, batch_loader = privacy_manager.privatize(noise_multiplier=sigma)

# Istanza dell'ottimizzatore privatizzato per il Client 0
optimizer_d0 = dpoptimizer_cls(local_discriminators[0].parameters(), lr=0.0002)

# --- FIX ATTRIBUTEERROR: INIZIALIZZAZIONE GRADIENTI ---
# Facciamo un passaggio "finto" per popolare i gradienti prima che il lot_loader li cerchi
dummy_x, _ = next(iter(client_loaders[0]))
dummy_x = dummy_x.to(device)
local_discriminators[0].zero_grad()
local_discriminators[0](dummy_x).mean().backward()
optimizer_d0.zero_grad()

In [11]:
from tqdm import tqdm

print("🏋️ Training avviato. Monitoraggio attivo...")
criterion = nn.BCELoss()

for epoch in range(epochs):
    # Usiamo tqdm per vedere la progressione reale dei lotti
    lot_pbar = tqdm(lot_loader(optimizer_d0), desc=f"Epoca {epoch+1}")
    
    for X_lot, y_lot in lot_pbar:
        # Debug rapido: stampa la dimensione del lotto solo la prima volta
        # print(f"Processando lotto di dimensione: {X_lot.size(0)}") 

        for X_batch, y_batch in batch_loader(TensorDataset(X_lot, y_lot)):
            optimizer_d0.zero_grad()
            X_batch = X_batch.to(device)
            
            # --- DISCRIMINATORE ---
            output_real = local_discriminators[0](X_batch)
            loss_real = criterion(output_real, torch.ones_like(output_real) * 0.9)
            
            noise = torch.randn(X_batch.size(0), lat_dim, 1, 1, device=device)
            fake_imgs = global_generator(noise)
            output_fake = local_discriminators[0](fake_imgs.detach())
            loss_fake = criterion(output_fake, torch.zeros_like(output_fake))
            
            (loss_real + loss_fake).backward()
            optimizer_d0.step()
            
            # --- GENERATORE ---
            gen_opt.zero_grad()
            output_g = local_discriminators[0](fake_imgs)
            loss_g = criterion(output_g, torch.ones_like(output_g))
            loss_g.backward()
            gen_opt.step()
            
        # Aggiorna la barra con l'epsilon attuale
        lot_pbar.set_postfix({"eps": f"{accountant.get_epsilon(delta=t_delta):.2f}"})

    print(f"✅ Epoca {epoch+1} terminata.")

🏋️ Training avviato. Monitoraggio attivo...


Epoca 1: 100%|██████████| 13220/13220 [2:26:26<00:00,  1.50it/s, eps=6.33] 


✅ Epoca 1 terminata.


Epoca 2:  13%|█▎        | 1772/13220 [29:59<3:13:45,  1.02s/it, eps=6.48]


KeyboardInterrupt: 